In [1]:
import os
import glob
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [2]:
TRAIN_DIR = "/kaggle/input/datasets/cheukhongtang/ass3data/train/train"
TEST_DIR = "/kaggle/input/datasets/cheukhongtang/ass3data/test/test"
SUB_PATH = "/kaggle/input/datasets/cheukhongtang/ass3data/sample_submission.csv"

train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "User_*", "*.csv")))
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "User_*", "*.csv")))

print("Train files:", len(train_files))
print("Test files:", len(test_files))

Train files: 11020
Test files: 6849


In [3]:
def get_user_from_path(path):
    parts = path.replace("\\", "/").split("/")
    for p in parts:
        if p.startswith("User_"):
            return p
    return "Unknown"


def extract_simple_features(path, is_train=True):
    df = pd.read_csv(path)
    df = df.sort_values("index").reset_index(drop=True)

    row = {}
    row["path"] = path
    row["user"] = get_user_from_path(path)

    if "file_id" in df.columns:
        row["file_id"] = int(df["file_id"].iloc[0])
    else:
        row["file_id"] = int(os.path.splitext(os.path.basename(path))[0])

    if is_train:
        row["label"] = int(df["label"].iloc[0])

    # Naive baseline:
    # Use ONLY the original raw axis columns and basic statistics.
    # No magnitude preprocessing is used here.
    simple_cols = [
        "mean_x", "mean_y", "mean_z",
        "std_x", "std_y", "std_z"
    ]

    for col in simple_cols:
        values = df[col].values

        row[f"{col}_mean"] = np.mean(values)
        row[f"{col}_std"] = np.std(values)
        row[f"{col}_min"] = np.min(values)
        row[f"{col}_max"] = np.max(values)
        row[f"{col}_range"] = np.max(values) - np.min(values)
        row[f"{col}_median"] = np.median(values)

    return row


def build_simple_feature_table(files, is_train=True):
    rows = []
    for path in tqdm(files):
        rows.append(extract_simple_features(path, is_train=is_train))
    return pd.DataFrame(rows)

In [4]:
simple_train = build_simple_feature_table(train_files, is_train=True)
simple_test = build_simple_feature_table(test_files, is_train=False)

print("Simple train shape:", simple_train.shape)
print("Simple test shape:", simple_test.shape)

display(simple_train.head())

  0%|          | 0/11020 [00:00<?, ?it/s]

  0%|          | 0/6849 [00:00<?, ?it/s]

Simple train shape: (11020, 40)
Simple test shape: (6849, 39)


,path,user,file_id,label,mean_x_mean,mean_x_std,mean_x_min,mean_x_max,mean_x_range,mean_x_median,...,std_y_min,std_y_max,std_y_range,std_y_median,std_z_mean,std_z_std,std_z_min,std_z_max,std_z_range,std_z_median
0,/kaggle/input/datasets/cheukhongtang/ass3data/...,User_001,1,0,-0.474944,0.003214,-0.480706,-0.466690,0.014015,-0.475184,...,0.0,0.008950,0.008950,0.004562,0.003545,0.002025,0.000000,0.007613,0.007613,0.003372
1,/kaggle/input/datasets/cheukhongtang/ass3data/...,User_001,2,0,-0.477411,0.002954,-0.482270,-0.466157,0.016113,-0.478000,...,0.0,0.008422,0.008422,0.006818,0.005853,0.001623,0.000000,0.007741,0.007741,0.006193
2,/kaggle/input/datasets/cheukhongtang/ass3data/...,User_001,3,0,-0.477836,0.002302,-0.481455,-0.472571,0.008884,-0.478405,...,0.0,0.007823,0.007823,0.005875,0.007399,0.000435,0.005525,0.008033,0.002508,0.007597
3,/kaggle/input/datasets/cheukhongtang/ass3data/...,User_001,4,0,-0.476572,0.002982,-0.481614,-0.469691,0.011923,-0.477044,...,0.0,0.008113,0.008113,0.003409,0.003517,0.002963,0.000000,0.009359,0.009359,0.002166
4,/kaggle/input/datasets/cheukhongtang/ass3data/...,User_001,5,0,-0.476293,0.002300,-0.480955,-0.471099,0.009855,-0.476830,...,0.0,0.007459,0.007459,0.003990,0.001567,0.001121,0.000000,0.004734,0.004734,0.001540


In [5]:
id_cols = ["path", "user", "file_id"]
target_col = "label"

simple_feature_cols = [
    c for c in simple_train.columns
    if c not in id_cols + [target_col]
]

X_simple = simple_train[simple_feature_cols].copy()
y_simple = simple_train[target_col].copy()
groups_simple = simple_train["user"].copy()

X_simple = X_simple.replace([np.inf, -np.inf], np.nan)
medians_simple = X_simple.median()
X_simple = X_simple.fillna(medians_simple)

sample_submission = pd.read_csv(SUB_PATH)

simple_test_ordered = sample_submission[["Id"]].merge(
    simple_test,
    left_on="Id",
    right_on="file_id",
    how="left"
)

print("Missing test matches:", simple_test_ordered["file_id"].isna().sum())

X_simple_test_ordered = simple_test_ordered[simple_feature_cols].copy()
X_simple_test_ordered = X_simple_test_ordered.replace([np.inf, -np.inf], np.nan)
X_simple_test_ordered = X_simple_test_ordered.fillna(medians_simple)

print("X_simple:", X_simple.shape)
print("X_simple_test_ordered:", X_simple_test_ordered.shape)

Missing test matches: 0
X_simple: (11020, 36)
X_simple_test_ordered: (6849, 36)


In [6]:
def evaluate_model_groupkfold(model_name, model, X, y, groups, use_sample_weight=True):
    gkf = GroupKFold(n_splits=5)
    
    oof_pred = np.zeros(len(X), dtype=int)
    fold_scores = []
    fold_accs = []
    models = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), start=1):
        print(f"\n{model_name} - Fold {fold}")
        
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        
        if use_sample_weight:
            sample_weight = compute_sample_weight(
                class_weight="balanced",
                y=y_train
            )
            model.fit(X_train, y_train, sample_weight=sample_weight)
        else:
            model.fit(X_train, y_train)
        
        pred = model.predict(X_val)
        
        if pred.ndim > 1:
            pred = pred.reshape(-1)
        
        pred = pred.astype(int)
        
        oof_pred[val_idx] = pred
        
        macro_f1 = f1_score(y_val, pred, average="macro")
        acc = accuracy_score(y_val, pred)
        
        fold_scores.append(macro_f1)
        fold_accs.append(acc)
        models.append(model)
        
        print(f"Macro-F1: {macro_f1:.4f}")
        print(f"Accuracy: {acc:.4f}")
    
    print("\n==============================")
    print(model_name)
    print("Mean macro-F1:", np.mean(fold_scores))
    print("Std macro-F1:", np.std(fold_scores))
    print("Mean accuracy:", np.mean(fold_accs))
    print("==============================")
    
    print(classification_report(y, oof_pred, digits=4))
    
    return {
        "name": model_name,
        "models": models,
        "oof_pred": oof_pred,
        "fold_macro_f1": fold_scores,
        "mean_macro_f1": np.mean(fold_scores),
        "std_macro_f1": np.std(fold_scores),
        "mean_accuracy": np.mean(fold_accs),
    }

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

gkf = GroupKFold(n_splits=5)

rf_oof = np.zeros(len(X_simple), dtype=int)
rf_models = []
rf_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_simple, y_simple, groups_simple), start=1):
    print(f"\nRandom Forest - Fold {fold}")
    
    model = clone(rf_base)
    
    X_train = X_simple.iloc[train_idx]
    X_val = X_simple.iloc[val_idx]
    y_train = y_simple.iloc[train_idx]
    y_val = y_simple.iloc[val_idx]
    
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    
    rf_oof[val_idx] = pred
    
    score = f1_score(y_val, pred, average="macro")
    rf_scores.append(score)
    rf_models.append(model)
    
    print("Macro-F1:", score)

print("\nRandom Forest simple baseline")
print("Mean macro-F1:", np.mean(rf_scores))
print("Std macro-F1:", np.std(rf_scores))
print(classification_report(y_simple, rf_oof, digits=4))


Random Forest - Fold 1
Macro-F1: 0.6016011265743005

Random Forest - Fold 2
Macro-F1: 0.672415129116275

Random Forest - Fold 3
Macro-F1: 0.6657566233771647

Random Forest - Fold 4
Macro-F1: 0.6522351342990481

Random Forest - Fold 5
Macro-F1: 0.6017965533421413

Random Forest simple baseline
Mean macro-F1: 0.6387609133417859
Std macro-F1: 0.03095203674365727
              precision    recall  f1-score   support

           0     0.9355    0.9658    0.9504      4643
           1     0.8494    0.8952    0.8717      4695
           2     0.4167    0.0279    0.0524       358
           3     0.6081    0.7332    0.6648       656
           4     0.8739    0.6831    0.7668       142
           5     0.7365    0.4943    0.5916       526

    accuracy                         0.8652     11020
   macro avg     0.7367    0.6333    0.6496     11020
weighted avg     0.8522    0.8652    0.8512     11020



In [8]:
import lightgbm as lgb
from sklearn.base import clone

lgb_base = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=6,
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

gkf = GroupKFold(n_splits=5)

lgb_oof = np.zeros(len(X_simple), dtype=int)
lgb_models = []
lgb_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_simple, y_simple, groups_simple), start=1):
    print(f"\nLightGBM - Fold {fold}")
    
    model = clone(lgb_base)
    model.set_params(random_state=42 + fold)
    
    X_train = X_simple.iloc[train_idx]
    X_val = X_simple.iloc[val_idx]
    y_train = y_simple.iloc[train_idx]
    y_val = y_simple.iloc[val_idx]
    
    sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )
    
    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight,
        eval_set=[(X_val, y_val)],
        eval_metric="multi_logloss",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=100)
        ]
    )
    
    pred = model.predict(X_val).astype(int)
    
    lgb_oof[val_idx] = pred
    
    score = f1_score(y_val, pred, average="macro")
    lgb_scores.append(score)
    lgb_models.append(model)
    
    print("Macro-F1:", score)

print("\nLightGBM simple baseline")
print("Mean macro-F1:", np.mean(lgb_scores))
print("Std macro-F1:", np.std(lgb_scores))
print(classification_report(y_simple, lgb_oof, digits=4))


LightGBM - Fold 1
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.510918
[200]	valid_0's multi_logloss: 0.476278
[300]	valid_0's multi_logloss: 0.485711
Early stopping, best iteration is:
[219]	valid_0's multi_logloss: 0.474787
Macro-F1: 0.6240391308844621

LightGBM - Fold 2
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.457106
[200]	valid_0's multi_logloss: 0.405955
[300]	valid_0's multi_logloss: 0.404518
Early stopping, best iteration is:
[258]	valid_0's multi_logloss: 0.40172
Macro-F1: 0.706517161707061

LightGBM - Fold 3
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.389042
[200]	valid_0's multi_logloss: 0.346821
[300]	valid_0's multi_logloss: 0.352206
Early stopping, best iteration is:
[209]	valid_0's multi_logloss: 0.346482
Macro-F1: 0.7204239809097922

LightGBM - Fold 4
Training until validation scores don't improve for 100 rounds
[100]	

In [9]:
from xgboost import XGBClassifier
from sklearn.base import clone

xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=6,
    n_estimators=800,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

gkf = GroupKFold(n_splits=5)

xgb_oof = np.zeros(len(X_simple), dtype=int)
xgb_models = []
xgb_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_simple, y_simple, groups_simple), start=1):
    print(f"\nXGBoost - Fold {fold}")
    
    model = clone(xgb_base)
    model.set_params(random_state=42 + fold)
    
    X_train = X_simple.iloc[train_idx]
    X_val = X_simple.iloc[val_idx]
    y_train = y_simple.iloc[train_idx]
    y_val = y_simple.iloc[val_idx]
    
    sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )
    
    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    pred = model.predict(X_val).astype(int)
    
    xgb_oof[val_idx] = pred
    
    score = f1_score(y_val, pred, average="macro")
    xgb_scores.append(score)
    xgb_models.append(model)
    
    print("Macro-F1:", score)

print("\nXGBoost simple baseline")
print("Mean macro-F1:", np.mean(xgb_scores))
print("Std macro-F1:", np.std(xgb_scores))
print(classification_report(y_simple, xgb_oof, digits=4))


XGBoost - Fold 1
[0]	validation_0-mlogloss:1.73917
[100]	validation_0-mlogloss:0.61381
[200]	validation_0-mlogloss:0.52599
[300]	validation_0-mlogloss:0.50107
[400]	validation_0-mlogloss:0.48601
[500]	validation_0-mlogloss:0.47721
[600]	validation_0-mlogloss:0.47305
[700]	validation_0-mlogloss:0.47220
[799]	validation_0-mlogloss:0.47417
Macro-F1: 0.62735266906238

XGBoost - Fold 2
[0]	validation_0-mlogloss:1.73802
[100]	validation_0-mlogloss:0.57713
[200]	validation_0-mlogloss:0.47378
[300]	validation_0-mlogloss:0.43590
[400]	validation_0-mlogloss:0.41759
[500]	validation_0-mlogloss:0.40640
[600]	validation_0-mlogloss:0.40129
[700]	validation_0-mlogloss:0.39901
[799]	validation_0-mlogloss:0.39779
Macro-F1: 0.7191186385963845

XGBoost - Fold 3
[0]	validation_0-mlogloss:1.73788
[100]	validation_0-mlogloss:0.52657
[200]	validation_0-mlogloss:0.40933
[300]	validation_0-mlogloss:0.37483
[400]	validation_0-mlogloss:0.35709
[500]	validation_0-mlogloss:0.34868
[600]	validation_0-mlogloss:0.34

In [10]:
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report

gkf = GroupKFold(n_splits=5)

cat_oof = np.zeros(len(X_simple), dtype=int)
cat_models = []
cat_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_simple, y_simple, groups_simple), start=1):
    print(f"\nCatBoost - Fold {fold}")
    
    X_train = X_simple.iloc[train_idx]
    X_val = X_simple.iloc[val_idx]
    y_train = y_simple.iloc[train_idx]
    y_val = y_simple.iloc[val_idx]
    
    sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )
    
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=800,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3.0,
        random_seed=42 + fold,
        verbose=100,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100
    )
    
    pred = model.predict(X_val).reshape(-1).astype(int)
    
    cat_oof[val_idx] = pred
    
    score = f1_score(y_val, pred, average="macro")
    cat_scores.append(score)
    cat_models.append(model)
    
    print("Macro-F1:", score)

print("\nCatBoost simple baseline")
print("Mean macro-F1:", np.mean(cat_scores))
print("Std macro-F1:", np.std(cat_scores))
print(classification_report(y_simple, cat_oof, digits=4))


CatBoost - Fold 1
0:	learn: 1.7303235	test: 1.7268514	best: 1.7268514 (0)	total: 99.6ms	remaining: 1m 19s
100:	learn: 0.6492340	test: 0.6853964	best: 0.6853964 (100)	total: 3.32s	remaining: 23s
200:	learn: 0.5012422	test: 0.6248965	best: 0.6248965 (200)	total: 6.54s	remaining: 19.5s
300:	learn: 0.4191959	test: 0.6019102	best: 0.6019102 (300)	total: 9.72s	remaining: 16.1s
400:	learn: 0.3600146	test: 0.5886579	best: 0.5886579 (400)	total: 12.9s	remaining: 12.8s
500:	learn: 0.3136614	test: 0.5760543	best: 0.5760543 (500)	total: 16s	remaining: 9.57s
600:	learn: 0.2762827	test: 0.5633029	best: 0.5633029 (600)	total: 19.2s	remaining: 6.36s
700:	learn: 0.2460105	test: 0.5522448	best: 0.5522448 (700)	total: 22.4s	remaining: 3.16s
799:	learn: 0.2218716	test: 0.5424549	best: 0.5423867 (798)	total: 25.6s	remaining: 0us

bestTest = 0.5423867483
bestIteration = 798

Shrink model to first 799 iterations.
Macro-F1: 0.5860445608447542

CatBoost - Fold 2
0:	learn: 1.7365711	test: 1.7303211	best: 1.730

In [11]:
baseline_results = pd.DataFrame({
    "model": ["Random Forest", "LightGBM", "XGBoost", "CatBoost"],
    "features": ["basic stats only, raw axes"] * 4,
    "validation": ["GroupKFold by user"] * 4,
    "macro_f1_mean": [
        np.mean(rf_scores),
        np.mean(lgb_scores),
        np.mean(xgb_scores),
        np.mean(cat_scores)
    ],
    "macro_f1_std": [
        np.std(rf_scores),
        np.std(lgb_scores),
        np.std(xgb_scores),
        np.std(cat_scores)
    ]
}).sort_values("macro_f1_mean", ascending=False)

display(baseline_results)

,model,features,validation,macro_f1_mean,macro_f1_std
2,XGBoost,"basic stats only, raw axes",GroupKFold by user,0.684108,0.040275
1,LightGBM,"basic stats only, raw axes",GroupKFold by user,0.675358,0.034484
3,CatBoost,"basic stats only, raw axes",GroupKFold by user,0.674168,0.049705
0,Random Forest,"basic stats only, raw axes",GroupKFold by user,0.638761,0.030952


In [12]:
import numpy as np
import pandas as pd

def make_submission_from_models(models, X_test, sample_submission, output_name, model_type="proba"):
    if model_type == "proba":
        test_proba = np.zeros((len(X_test), 6))

        for model in models:
            test_proba += model.predict_proba(X_test) / len(models)

        test_pred = np.argmax(test_proba, axis=1)

    else:
        fold_preds = []

        for model in models:
            pred = model.predict(X_test)
            pred = np.asarray(pred).reshape(-1).astype(int)
            fold_preds.append(pred)

        fold_preds = np.vstack(fold_preds)

        test_pred = []
        for i in range(fold_preds.shape[1]):
            values, counts = np.unique(fold_preds[:, i], return_counts=True)
            test_pred.append(values[np.argmax(counts)])

        test_pred = np.array(test_pred)

    submission = sample_submission.copy()
    submission["Label"] = test_pred.astype(int)

    print("\n", output_name)
    print("Shape:", submission.shape)
    print("Prediction distribution:")
    display(submission["Label"].value_counts().sort_index())

    submission.to_csv(output_name, index=False)
    print("Saved:", output_name)

    return submission

In [13]:
submission_rf = make_submission_from_models(
    rf_models,
    X_simple_test_ordered,
    sample_submission,
    "submission_basic_stats_random_forest.csv"
)

submission_lgb = make_submission_from_models(
    lgb_models,
    X_simple_test_ordered,
    sample_submission,
    "submission_basic_stats_lightgbm.csv"
)

submission_xgb = make_submission_from_models(
    xgb_models,
    X_simple_test_ordered,
    sample_submission,
    "submission_basic_stats_xgboost.csv"
)

submission_cat = make_submission_from_models(
    cat_models,
    X_simple_test_ordered,
    sample_submission,
    "submission_basic_stats_catboost.csv"
)


 submission_basic_stats_random_forest.csv
Shape: (6849, 2)
Prediction distribution:


Label
0    2899
1    3134
2       6
3     550
4      50
5     210
Name: count, dtype: int64

Saved: submission_basic_stats_random_forest.csv

 submission_basic_stats_lightgbm.csv
Shape: (6849, 2)
Prediction distribution:


Label
0    2865
1    2960
2     144
3     540
4      74
5     266
Name: count, dtype: int64

Saved: submission_basic_stats_lightgbm.csv

 submission_basic_stats_xgboost.csv
Shape: (6849, 2)
Prediction distribution:


Label
0    2853
1    2983
2     134
3     544
4      65
5     270
Name: count, dtype: int64

Saved: submission_basic_stats_xgboost.csv

 submission_basic_stats_catboost.csv
Shape: (6849, 2)
Prediction distribution:


Label
0    2890
1    2712
2     263
3     589
4      69
5     326
Name: count, dtype: int64

Saved: submission_basic_stats_catboost.csv


In [14]:
import os

for file in [
    "submission_basic_stats_random_forest.csv",
    "submission_basic_stats_lightgbm.csv",
    "submission_basic_stats_xgboost.csv",
    "submission_basic_stats_catboost.csv"
]:
    print(file, os.path.exists(file))

submission_basic_stats_random_forest.csv True
submission_basic_stats_lightgbm.csv True
submission_basic_stats_xgboost.csv True
submission_basic_stats_catboost.csv True
